In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset, random_split

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

set_seed(42)
device = get_device()
print(f"Device: {device}")

In [ ]:
def get_dataloaders(data_dir='./data', batch_size=128, seed=42):
    set_seed(seed)
    os.makedirs(data_dir, exist_ok=True)
    transform = transforms.Compose([
        transforms.ToTensor()
    ])
    train_raw = datasets.FashionMNIST(root=data_dir, train=True, download=True, transform=transform)
    test_raw = datasets.FashionMNIST(root=data_dir, train=False, download=True, transform=transform)
    full_data = ConcatDataset([train_raw, test_raw])
    total = len(full_data)
    train_size = int(0.8 * total)
    val_size = int(0.1 * total)
    test_size = total - train_size - val_size
    gen = torch.Generator().manual_seed(seed)
    train_set, val_set, test_set = random_split(full_data, [train_size, val_size, test_size], generator=gen)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_dataloaders(batch_size=128, seed=42)
print("Data loaded successfully.")

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=64, num_classes=10):
        super(MLPClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, latent_dim)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(latent_dim, num_classes)

    def extract_latent(self, x):
        x = x.view(x.size(0), -1)
        h1 = self.relu1(self.fc1(x))
        z = self.relu2(self.fc2(h1))
        return z

    def forward(self, x):
        z = self.extract_latent(x)
        logits = self.fc3(z)
        return logits

def evaluate_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

In [ ]:
model = MLPClassifier().to(device)
model_path = 'checkpoints/mlp_classifier.pth'

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print("Loaded weights from", model_path)
else:
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    epochs = 15
    best_val_acc = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
        train_loss = running_loss / total_train
        train_acc = correct_train / total_train
        val_acc = evaluate_accuracy(model, val_loader, device)
        print(f"Epoch [{epoch}/{epochs}] - Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            os.makedirs('checkpoints', exist_ok=True)
            torch.save(model.state_dict(), model_path)
    model.load_state_dict(torch.load(model_path, map_location=device))

test_acc = evaluate_accuracy(model, test_loader, device)
print(f"Final Test Set Accuracy: {test_acc*100:.2f}% (Target: >=88.00%)")

In [ ]:
W1 = model.fc1.weight.detach()
W2 = model.fc2.weight.detach()
W3 = model.fc3.weight.detach()

relu = nn.ReLU()
sigmoid = nn.Sigmoid()

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for c in range(10):
    yc = torch.zeros(1, 10, device=device)
    yc[0, c] = 1.0
    z_hat = relu(torch.matmul(yc, W3))
    h1_hat = relu(torch.matmul(z_hat, W2))
    x_hat = sigmoid(torch.matmul(h1_hat, W1))
    img = x_hat.view(28, 28).cpu().numpy()
    
    axes[c].imshow(img, cmap='gray')
    axes[c].set_title(CLASS_NAMES[c], fontsize=8)
    axes[c].axis('off')

plt.suptitle('Tahap 2.2: Proyeksi Balik Bobot Kelas (W^T)', fontsize=12)
plt.show()

In [ ]:
data_iter = iter(test_loader)
real_images, real_labels = next(data_iter)
real_images = real_images[:8].to(device)

with torch.no_grad():
    z_mlp = model.extract_latent(real_images)
    h1_proj = relu(torch.matmul(z_mlp, W2))
    x_proj = sigmoid(torch.matmul(h1_proj, W1))

real_imgs = real_images.cpu().numpy()
proj_imgs = x_proj.view(-1, 28, 28).cpu().numpy()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(real_imgs[i, 0], cmap='gray')
    axes[0, i].set_title(CLASS_NAMES[real_labels[i].item()], fontsize=8)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(proj_imgs[i], cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=9)
axes[1, 0].set_ylabel('W^T Proj', fontsize=9)
plt.suptitle('Tahap 2.3: Proyeksi Balik Fitur Laten z_MLP via W^T vs Citra Asli', fontsize=11)
plt.show()

In [ ]:
class FrozenLatentDecoder(nn.Module):
    def __init__(self, latent_dim=64, hidden_dim=256, output_dim=784):
        super(FrozenLatentDecoder, self).__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, z):
        h = self.relu1(self.fc1(z))
        out = self.sigmoid(self.fc2(h))
        return out.view(-1, 1, 28, 28)

for param in model.parameters():
    param.requires_grad = False

decoder = FrozenLatentDecoder().to(device)
criterion_mse = nn.MSELoss()
optimizer_dec = torch.optim.Adam(decoder.parameters(), lr=0.001)
decoder_path = 'checkpoints/frozen_latent_decoder.pth'
decoder_epochs = 15

if os.path.exists(decoder_path):
    decoder.load_state_dict(torch.load(decoder_path, map_location=device))
    print("Loaded decoder weights from", decoder_path)
else:
    for ep in range(1, decoder_epochs + 1):
        decoder.train()
        total_loss = 0.0
        n_samples = 0
        for imgs, _ in train_loader:
            imgs = imgs.to(device)
            optimizer_dec.zero_grad()
            with torch.no_grad():
                z = model.extract_latent(imgs)
            rec = decoder(z)
            loss = criterion_mse(rec, imgs)
            loss.backward()
            optimizer_dec.step()
            total_loss += loss.item() * imgs.size(0)
            n_samples += imgs.size(0)
        avg_loss = total_loss / n_samples
        print(f"Decoder Epoch [{ep}/{decoder_epochs}] - MSE Loss: {avg_loss:.5f}")
    os.makedirs('checkpoints', exist_ok=True)
    torch.save(decoder.state_dict(), decoder_path)

In [ ]:
decoder.eval()
with torch.no_grad():
    z_test = model.extract_latent(real_images)
    recon_images = decoder(z_test)

recon_np = recon_images.cpu().numpy()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(real_imgs[i, 0], cmap='gray')
    axes[0, i].set_title(CLASS_NAMES[real_labels[i].item()], fontsize=8)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(recon_np[i, 0], cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=9)
axes[1, 0].set_ylabel('Frozen Dec', fontsize=9)
plt.suptitle('Tahap 2.4: Rekonstruksi dari Frozen Latent Decoder', fontsize=11)
plt.show()

In [ ]:
all_latents = []
model.eval()
with torch.no_grad():
    for imgs, _ in test_loader:
        imgs = imgs.to(device)
        z = model.extract_latent(imgs)
        all_latents.append(z)

all_latents = torch.cat(all_latents, dim=0)
mu_z = torch.mean(all_latents, dim=0)
sigma_z = torch.std(all_latents, dim=0)

n_samples = 16
eps = torch.randn(n_samples, 64, device=device)
z_rand = mu_z + eps * sigma_z

decoder.eval()
with torch.no_grad():
    fake_samples = decoder(z_rand).cpu().numpy()

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i in range(16):
    r, c = divmod(i, 4)
    axes[r, c].imshow(fake_samples[i, 0], cmap='gray')
    axes[r, c].axis('off')

plt.suptitle('Tahap 2.5: Uji Sampling Generatif pada Ruang Laten MLP', fontsize=11)
plt.tight_layout()
plt.show()